# MCNN-Eff-LST on CIFAR-10

End-to-end study notebook for the **MCNN-Eff-LST** model. The notebook is organised into eight self-contained sections — once sections 1–4 have been executed, the remaining sections can be run independently.

1. Imports & device setup
2. Dataset (CIFAR-10)
3. Model: `EffNet_LST`
4. Training / evaluation helpers
5. Load pretrained model and evaluate on the test set
6. Train a single model with custom hyperparameters
7. Hyperparameter search with Optuna
8. 10-run statistics with the best configuration


## 1. Imports & device setup

In [1]:
import gc
import os
import shutil

import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.optim import lr_scheduler
from torch.nn.functional import max_pool2d, relu
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

from l2dst_lib.lst_nn import L2DST, multichan_to_2D
from EfficientNet_lib import MBConvBlock

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  |  device: {device}")


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


PyTorch 2.1.2+cpu  |  device: cpu


## 2. Dataset (CIFAR-10)

CIFAR-10 is downloaded automatically the first time the cell runs. The full 50 000-image training set is used for training; the held-out 10 000-image test set is used both for validation during training and for final evaluation, following the protocol of the paper.

In [2]:
Dataset_name = 'CIFAR10'
DATA_DIR = '../Datasets'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset_full = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=transform)

# CIFAR-10 protocol used in the paper: the test set doubles as validation.
val_loader  = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)
test_loader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)
print(f"train={len(trainset_full)}  test={len(testset)}")


Files already downloaded and verified
Files already downloaded and verified
train=50000  test=10000


## 3. Model: `EffNet_LST`

Four parallel `Conv2d` columns extract multi-scale features that are concatenated, refined by three EfficientNet-style `MBConvBlock` stages, and finally aggregated by a single `LST` block (used here in place of global average pooling).

**Constraint:** `mbconv_channels` must be a perfect square — the channel dimension is reshaped into a 2-D grid by `multichan_to_2D` before LST.

In [3]:
class EffNet_LST(nn.Module):
    def __init__(self, input_channel=3, c1_kernels=16, mbconv_channels=16,
                 lst_out=4, num_classes=10,
                 p_drop_lst=0.1, p_drop_fc=0.1, device='cpu'):
        super().__init__()
        image_size = 32
        self.conv1 = nn.Conv2d(input_channel, c1_kernels, 2, padding='same')
        self.conv2 = nn.Conv2d(input_channel, c1_kernels, 3, padding='same')
        self.conv3 = nn.Conv2d(input_channel, c1_kernels, 4, padding='same')
        self.conv4 = nn.Conv2d(input_channel, c1_kernels, 5, padding='same')

        total_concat_channels = 4 * c1_kernels
        self.mbconv1 = MBConvBlock(total_concat_channels, 16, kernel_size=3, stride=1, expand_ratio=1)
        self.mbconv2 = MBConvBlock(16, 24, kernel_size=3, stride=1, expand_ratio=6)
        self.mbconv3 = MBConvBlock(24, mbconv_channels, kernel_size=3, stride=1, expand_ratio=6)

        self.BN = nn.BatchNorm2d(1)
        self.lst_out = lst_out

        s = int(np.sqrt(mbconv_channels))
        assert s * s == mbconv_channels, "mbconv_channels must be a perfect square"

        self.lst = L2DST([(image_size // 2) * s, (image_size // 2) * s],
                         [lst_out, lst_out], device=device, p_drop=p_drop_lst)

        self.fc1 = nn.Linear(lst_out * lst_out, num_classes)
        self.drop1 = nn.Dropout(p=p_drop_fc)
        self._initialize_weights()

    def _initialize_weights(self):
        for conv in [self.conv1, self.conv2, self.conv3, self.conv4]:
            nn.init.xavier_normal_(conv.weight, gain=nn.init.calculate_gain('relu'))
            if conv.bias is not None:
                nn.init.constant_(conv.bias, 0)
        self.mbconv1._initialize_weights()
        self.mbconv2._initialize_weights()
        self.mbconv3._initialize_weights()
        nn.init.xavier_normal_(self.fc1.weight)
        if self.fc1.bias is not None:
            nn.init.constant_(self.fc1.bias, 0)

    def forward(self, x):
        x1 = max_pool2d(relu(self.conv1(x)), (2, 2))
        x2 = max_pool2d(relu(self.conv2(x)), (2, 2))
        x3 = max_pool2d(relu(self.conv3(x)), (2, 2))
        x4 = max_pool2d(relu(self.conv4(x)), (2, 2))

        out = torch.cat([x1, x2, x3, x4], dim=1)
        out = self.mbconv1(out)
        out = self.mbconv2(out)
        out = self.mbconv3(out)

        out = multichan_to_2D(out)
        out = self.lst(out)
        out = self.BN(out.unsqueeze(1))
        out = torch.flatten(out, 1)
        return self.fc1(self.drop1(out))


# Quick sanity check
_m = EffNet_LST(input_channel=3, c1_kernels=16, mbconv_channels=36,
                lst_out=32, device=device).to(device)
_n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"EffNet_LST(c1=16, mb=36, lst_out=32) parameters = {_n_params:,}")
del _m


EffNet_LST(c1=16, mb=36, lst_out=32) parameters = 43,894


## 4. Training & evaluation helpers

In [4]:
def acc_estimate(model, data_loader):
    """Compute classification accuracy of `model` on `data_loader`."""
    model.to(device)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y_true in data_loader:
            X = X.to(device)
            y_true = y_true.to(device)
            outputs = model(X)
            _, y_pred = torch.max(outputs, dim=1)
            total += y_true.shape[0]
            correct += int((y_pred == y_true).sum())
    return correct / total


def remove_directory(dir_path):
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)


def train_test_loop(model, model_name, Dataset_name, trainloader, val_loader,
                    test_loader=None, N_epoch=200, weight_decay=1e-6, lr=2e-3,
                    T0=100, eta_min=1e-6, save_best=True):
    """Train `model` and track best accuracy on `val_loader`.

    The checkpoint with the highest validation accuracy is written to
    `model_backup/{Dataset_name}/{model_name}/{model_name}_epoch_{N_epoch}.pth`.
    Returns (best_val_acc, test_acc) where `test_acc` is computed on
    `test_loader` if provided, otherwise equals `best_val_acc`.
    """
    model_path = os.path.join('model_backup', Dataset_name, model_name)
    os.makedirs(model_path, exist_ok=True)
    target_path = os.path.join(model_path, f"{model_name}_epoch_{N_epoch}.pth")

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr,
                            weight_decay=weight_decay, amsgrad=True)
    scheduler = lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=1, eta_min=eta_min)

    best_acc = 0.0
    for epoch in range(N_epoch):
        # -- validation --
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                output = model(images)
                val_loss += criterion(output, labels).item()
                _, y_pred = torch.max(output, dim=1)
                total += labels.shape[0]
                correct += int((y_pred == labels).sum())
        val_loss /= len(val_loader)
        val_acc = correct / total
        writer.add_scalar('Accuracy (val)', val_acc, epoch)
        writer.add_scalar('Val loss', val_loss, epoch)

        if save_best and val_acc > best_acc:
            torch.save(model.state_dict(), target_path)
            best_acc = val_acc

        # -- training --
        model.train()
        running_loss = 0.0
        for images, labels in trainloader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        writer.add_scalar('Train loss', running_loss / len(trainloader), epoch)

    if save_best and os.path.exists(target_path):
        model.load_state_dict(torch.load(target_path, map_location=device))
    test_acc = acc_estimate(model, test_loader) if test_loader is not None else best_acc
    print(f"Best val acc = {best_acc:.4f},  test acc = {test_acc:.4f}")
    return best_acc, test_acc


## 5. Load pretrained model and evaluate on the test set

Loads the checkpoint shipped under `model_backup/CIFAR10/` and reports its accuracy on the CIFAR-10 test set.

In [7]:
# Architecture hyperparameters of the released checkpoint
KERNEL_NUM     = 16
MBCONV_CH_NUM  = 36
LST_OUT        = 24
P_DROP_FC      = 0.10
P_DROP_LST     = 0.10

CHECKPOINT_NAME = 'MCNN_Eff_LST24_ker16_mbCh36'
CHECKPOINT_PATH = os.path.join(
    'pretrain', f'{CHECKPOINT_NAME}.pth')

print(f"Loading {CHECKPOINT_PATH}")
model = EffNet_LST(
    input_channel=3, c1_kernels=KERNEL_NUM,
    mbconv_channels=MBCONV_CH_NUM, lst_out=LST_OUT,
    p_drop_fc=P_DROP_FC, p_drop_lst=P_DROP_LST, device=device,
).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
acc_test = acc_estimate(model, test_loader)
print(f"Parameters    = {n_params:,}")
print(f"Test accuracy = {acc_test:.4f}")


Loading pretrain\MCNN_Eff_LST24_ker16_mbCh36.pth


c:\Users\m.vashkevich\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ..\aten\src\ATen\native\Convolution.cpp:1009.)
  return F.conv2d(input, weight, bias, self.stride,


Parameters    = 37,862
Test accuracy = 0.8428


## 6. Train a single model with custom hyperparameters

Adjust the constants below and run the cell to train one model from scratch. The best checkpoint (by test-set accuracy) is written to `model_backup/CIFAR10/<MODEL_NAME>/`.

In [ ]:
# --- user-tunable hyperparameters (uncomment to override) ----------------
LR             = 0.008435231056183082
WD             = 1.1059192437247107e-10
P_DROP_FC      = 0.16143306280558967
P_DROP_LST     = 0.48833363796528245
T_0_LOOPS      = 4
LST_OUT        = 24
KERNEL_NUM     = 16
MBCONV_CH_NUM  = 36
BATCH_SIZE     = 1024
N_EPOCHS       = 200
ETA_MIN        = LR * 0.01
RANDOM_SEED    = 27
# --------------------------------------------------------------------------


torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_name = f'EffNet_LST{LST_OUT}_ker{KERNEL_NUM}_mbCh{MBCONV_CH_NUM}_custom'
model = EffNet_LST(
    input_channel=3, c1_kernels=KERNEL_NUM,
    mbconv_channels=MBCONV_CH_NUM, lst_out=LST_OUT,
    p_drop_fc=P_DROP_FC, p_drop_lst=P_DROP_LST, device=device,
).to(device)

T_0 = N_EPOCHS // T_0_LOOPS
TENSORBOARD_LOGS_DIR = (
    f"runs/{model_name}/lr{LR:1.2e}-wd{WD:1.2e}-T0_{T_0}")
writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

trainloader = torch.utils.data.DataLoader(
    trainset_full, batch_size=BATCH_SIZE, shuffle=True)

best_val, acc_test = train_test_loop(
    model, model_name, Dataset_name,
    trainloader, val_loader, test_loader=test_loader,
    N_epoch=N_EPOCHS, weight_decay=WD, lr=LR,
    T0=T_0, eta_min=ETA_MIN)
writer.close()
print(f"Saved best checkpoint under model_backup/CIFAR10/{model_name}/")

## 7. Hyperparameter search with Optuna

The search space matches the one used in the paper. The shipped `db_cifar.sqlite3` already contains the trials from the study referenced in the paper — `load_if_exists=True` lets you either inspect that study or extend it with more trials.

In [ ]:
import optuna

SQLITE_STORAGE_URL = "sqlite:///db_cifar.sqlite3"
STUDY_NAME         = "j-iet-EffNetLST-CIFAR10"
N_TRIALS           = 100
N_EPOCHS_OPTUNA    = 200


def objective(trial: optuna.Trial) -> float:
    lr            = trial.suggest_float("lr", 6e-4, 1e-2, log=True)
    wd            = trial.suggest_float("wd", 5e-12, 1e-8, log=True)
    p_drop_fc     = trial.suggest_float("p_drop_fc",  0.10, 0.60)
    p_drop_lst    = trial.suggest_float("p_drop_lst", 0.09, 0.55)
    t_0_loop      = trial.suggest_categorical("t_0_loop", [10, 4, 2])
    lst_out       = trial.suggest_categorical("lst_out", [32, 48])
    kernel_num    = trial.suggest_categorical("kernel_num", [8, 12, 16, 36, 48])
    mbconv_ch_num = trial.suggest_categorical("mbconv_ch_num", [36, 49, 64])

    batch_size = 1024
    eta_min    = lr * 0.01

    torch.manual_seed(27)
    np.random.seed(27)

    model_name = f'EffNet_LST{lst_out}_ker{kernel_num}_mbCh{mbconv_ch_num}_t{trial.number}'
    model = EffNet_LST(
        input_channel=3, c1_kernels=kernel_num,
        mbconv_channels=mbconv_ch_num, lst_out=lst_out,
        p_drop_fc=p_drop_fc, p_drop_lst=p_drop_lst, device=device,
    ).to(device)

    T_0 = N_EPOCHS_OPTUNA // t_0_loop

    global TENSORBOARD_LOGS_DIR, writer
    TENSORBOARD_LOGS_DIR = (
        f"runs/optuna/cifar/{model_name}-lr{lr:1.2e}-wd{wd:1.2e}-T0_{T_0}")
    writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

    trainloader = torch.utils.data.DataLoader(
        trainset_full, batch_size=batch_size, shuffle=True)
    best_val, _ = train_test_loop(
        model, model_name, Dataset_name,
        trainloader, val_loader, test_loader=None,
        N_epoch=N_EPOCHS_OPTUNA, weight_decay=wd, lr=lr,
        T0=T_0, eta_min=eta_min)
    writer.close()
    torch.cuda.empty_cache()
    gc.collect()
    return best_val


study = optuna.create_study(
    storage=SQLITE_STORAGE_URL,
    study_name=STUDY_NAME,
    direction='maximize',
    load_if_exists=True,
)
print(f"Study contains {len(study.trials)} trial(s).")


In [ ]:
# Launch / resume the search (set N_TRIALS to 0 above to just inspect the study)
study.optimize(objective, n_trials=N_TRIALS)
print("Best params:", study.best_params)
print(f"Best value : {study.best_value:.4f}")


## 8. 10-run statistics with the best configuration

Trains the best configuration ten times with different RNG seeds and reports mean ± standard deviation of test accuracy.

In [ ]:
import optuna

N_RUNS       = 10
N_EPOCHS_RUN = 200
BATCH_SIZE   = 1024

SQLITE_STORAGE_URL = "sqlite:///db_cifar.sqlite3"
STUDY_NAME         = "j-iet-EffNetLST-CIFAR10"

try:
    study = optuna.load_study(study_name=STUDY_NAME, storage=SQLITE_STORAGE_URL)
    best_params = study.best_params
    LR_BEST           = best_params["lr"]
    WD_BEST           = best_params["wd"]
    P_DROP_FC_BEST    = best_params["p_drop_fc"]
    P_DROP_LST_BEST   = best_params["p_drop_lst"]
    T_0_LOOP_BEST     = best_params["t_0_loop"]
    KERNEL_NUM_BEST   = best_params["kernel_num"]
    MBCONV_CH_BEST    = best_params["mbconv_ch_num"]
    LST_OUT_BEST      = best_params["lst_out"]
    print("Loaded best params from Optuna study.")
except Exception as e:
    print(f"Falling back to checkpoint hyperparameters ({e}).")
    LR_BEST, WD_BEST   = 2.0e-03, 1.0e-09
    P_DROP_FC_BEST     = 0.20
    P_DROP_LST_BEST    = 0.20
    T_0_LOOP_BEST      = 2
    KERNEL_NUM_BEST    = 16
    MBCONV_CH_BEST     = 36
    LST_OUT_BEST       = 32

ETA_MIN = LR_BEST * 0.01
T_0     = N_EPOCHS_RUN // T_0_LOOP_BEST

HEADER_FMT = "| {:36} | {:>7} | {:>6} |"
SEP_LINE   = "|" + "-" * 38 + "|" + "-" * 9 + "|" + "-" * 8 + "|"

results_path = (f'results_EffNet_LST-{LST_OUT_BEST}-k{KERNEL_NUM_BEST}'
                f'_mbconv{MBCONV_CH_BEST}.txt')
acc_list = []
with open(results_path, 'w') as f:
    print(HEADER_FMT.format("model_name", "params", "ACC"), file=f)
    print(SEP_LINE, file=f)
    for run in range(N_RUNS):
        torch.manual_seed(27 + run)
        np.random.seed(27 + run)

        model_name = (f'EffNet_LST{LST_OUT_BEST}_ker{KERNEL_NUM_BEST}'
                      f'_mbCh{MBCONV_CH_BEST}_run{run}')
        model = EffNet_LST(
            input_channel=3, c1_kernels=KERNEL_NUM_BEST,
            mbconv_channels=MBCONV_CH_BEST, lst_out=LST_OUT_BEST,
            p_drop_fc=P_DROP_FC_BEST, p_drop_lst=P_DROP_LST_BEST,
            device=device,
        ).to(device)

        if run == 0:
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        TENSORBOARD_LOGS_DIR = f"runs/stat/cifar/{model_name}"
        remove_directory(TENSORBOARD_LOGS_DIR)
        writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

        trainloader = torch.utils.data.DataLoader(
            trainset_full, batch_size=BATCH_SIZE, shuffle=True)
        _, acc = train_test_loop(
            model, model_name, Dataset_name,
            trainloader, val_loader, test_loader=test_loader,
            N_epoch=N_EPOCHS_RUN, weight_decay=WD_BEST, lr=LR_BEST,
            T0=T_0, eta_min=ETA_MIN)
        writer.close()
        acc_list.append(acc)
        line = f"| {model_name:36} | {n_params:7,} | {acc:6.4f} |"
        print(line, file=f)
        print(line)

    acc_mean = float(np.mean(acc_list))
    acc_std  = float(np.std(acc_list))
    summary = (f"| FINAL  mean +/- std (n={N_RUNS:2d})              "
               f"| {n_params:7,} | {acc_mean:6.4f} +/- {acc_std:6.4f} |")
    print(summary, file=f)
    print(summary)
print(f"Results written to {results_path}")